# LangGraph 

LangGrap is a workflow orchestration framework for building AI related workflows like agents and all 
we can use python also do these things but that will too complex
langchain also supports workflows, but langgraph has human in the loop using interupt, retries, ...ect which makes most suitable for AI workflows

### AI related tasks 
conversation history
tool results
retries
errors
parallel execution
human approval
persistence (short term memory,long term memory)
pause and resume
multiple agents

## use langraph when you need 
Dynamic routing
Cycles / loops
Multiple possible execution paths
State shared across steps
Persistence
Pause and resume
Human approval
Complex agent behavior

LangGraph is a framework for building applications as stateful graphs, where nodes perform work, edges control execution, and state carries information through the system.

In [ ]:
### components of Graph 

State : State is the shared data that represents the current condition of the graph.
Node : functions which takes state as input and modify the state and return state or part of it 
edge : linkage between nodes it will say after the current node where to go 
building graph  - we first define state, build nodes  and edges then for the graph we add edges and we complie the graph to use it 


if not state we have manually pass the output of each function to other function which is okay in normal linear workflow but when we have parallel workflows both nodes takes same input and produces the different so we need to manually join them


State is not a globale memory 
that is like an argument we send to graph and it modifies in it's entire worklfow and returns the updated state 

In [ ]:
### creating a state 
from typing import TypedDict


class State(TypedDict):
    query: str
    result: str
    retry_count: int

In [ ]:
## creating nodes 
def add_ten(state):
    return {
        "number": state["number"] + 10
    }

In [ ]:
### create a graph 

graph = StateGraph(State)  #Create a graph whose nodes operate on the State structure.


## step- 1 :  add nodes 
graph.add_node("add_ten", add_ten)    ## first argument will be node name and second will be actuall node 

graph.add_node(
    "multiply_by_two",
    multiply_by_two
)


## step-2 : add edges 
graph.add_edge(
    START,
    "add_ten"
)

graph.add_edge(
    "multiply_by_two",
    END
)

## complie  graph  - here we actually created an object of the blueprint we have created 
workflow = graph.compile()


#invoke graph - actually using it 
res = workflow.invoke(
    query = "",
    result = "",
    retry_count = ""
)

## for displaying the graph 
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

## Data FLow vs Control Flow 

Data Flow - What information exists? This is handled by state
COntrol Flow - Where should execution go next? This is handled by edges 

## Conditional Routing 
from a node when there are multiple possibile nodes to go, we need to route to particular node based on current node output 
for this conditionall routing we can create another node which does this selection based on current node output and return the node name to go futhur 


### Parallel Execution & Reducer
when we want two/more nodes to perform parallely 
here in this case nodes which are parallel should not return entire state just part of it then grpah merges all those into single state updates




### Reducer 
If multiple updates happen to the same state field, how should they be combined?
to implement reducer we use operator from python 

This happens in two scenerios 
1. when  two or more nodes which are in parallel are trying to modify the same key in the state, for this we need to specifiy what needs to happen
2. suppose we have agent loop 
User
 ↓
Agent
 ↓
Tool
 ↓
Agent
 ↓
Tool
 ↓
Agent

Each step needs to add a new message.

initial state 
messages = [
    user_message
]

agent produces 
[
    ai_message
]


tool produces 
[
    tool_message
]

again agent produces 
[
    another_ai_message
]

you dont want 
Old messages
REPLACED
by new message

you want 
Old messages
+
New message


### inetegrating tools 

for each tool we need to define proper name,arguments and docstrings for what it does such the agent can pick tool based on in it's functionality 

in langraph we use @tool decorator to specify something as tool 
we have bind_tools() which gives information about all tools to the agent 




### MessageState 

so we have seen requirement for reducer in message field of a state
Because message accumulation is so common, LangGraph provides a convenient predefined state called MessageState, it has 

##### 1. Human Message : Represents inputs from a user

##### 2. System Message : Used to Provide instructions or context to the model

##### 3. Function Message : Represents the result of a function call

##### 4. AI Message : Represents responses generated by AI models

##### 5. Tool Message : Similar to Function message, but specific to tool usage

### ToolNode
we have ToolNode which calls the requested tool and updates the MessageState with ToolMessage field with tool output


In [ ]:
from langchain_core.tools import tool
from langgraph.graph import (
    StateGraph,
    START,
    END,
    MessagesState
)
from langgraph.prebuilt import ToolNode


# -------------------------
# 1. Define Tool
# -------------------------

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""

    return a * b


tools = [multiply]


# -------------------------
# 2. Bind tools to LLM
# -------------------------

llm_with_tools = llm.bind_tools(tools)


# -------------------------
# 3. Agent Node
# -------------------------

def agent(state: MessagesState):

    response = llm_with_tools.invoke(
        state["messages"]
    )

    return {
        "messages": [response]
    }


# -------------------------
# 4. Routing Function
# -------------------------

def should_continue(state: MessagesState):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END


# -------------------------
# 5. Tool Node
# -------------------------

tool_node = ToolNode(tools)


# -------------------------
# 6. Create Graph
# -------------------------

graph = StateGraph(MessagesState)


# -------------------------
# 7. Add Nodes
# -------------------------

graph.add_node(
    "agent",
    agent
)

graph.add_node(
    "tools",
    tool_node
)


# -------------------------
# 8. Add START edge
# -------------------------

graph.add_edge(
    START,
    "agent"
)


# -------------------------
# 9. Conditional Edge
# -------------------------

graph.add_conditional_edges(
    "agent",
    should_continue
)


# -------------------------
# 10. Loop
# -------------------------

graph.add_edge(
    "tools",
    "agent"
)


# -------------------------
# 11. Compile
# -------------------------

app = graph.compile()

### Memorey
we have 
short term memory - lives in that application no external db is required . example chat history conversations
long term memory  - stored in db and fecthed from there ex : compliance or documents or an organization 

State is the information available during the current graph execution. this is not the short term thing 

for persistance during any failure of graph we have checkpointer which is like snapshot of the state at that moment 
while mulitple or current users, each has their own state and say we are storing the checkpointer for them but to identify which checkpoint belong which user we need also pass a thread id 

Checkpointing primarily means:

Save the state of graph execution so the graph can maintain/resume state. it is used for 

defination : A checkpointer in LangGraph persists graph state at execution points, allowing stateful conversations, recovery, resumption, and human-in-the-loop workflows. A thread ID identifies the execution/conversation history to which those checkpoints belong.

resume
failure recovery
threads
human-in-the-loop(when we pass for human resume and we resume after we got it )

The Most Important Part: Human-in-the-Loop

This is where checkpointing becomes particularly powerful.

Suppose your agent can make purchases.

You don't want:

User:
Buy laptop

Agent:
Okay → immediately purchase

Instead:

User
 ↓
Agent
 ↓
Purchase requested
 ↓
PAUSE
 ↓
Human approval
 ↓
Continue
 ↓
Purchase

The graph needs to remember:

Where was I?

What was the state?

What was the pending action?

Checkpointing makes this possible.

### subgraph 
when we are doing multiagent systems, each node we can treat as agent but under agent we have many steps so instead of one node
we define as graph as use it as node in the main workflow 

### Interrupts
This is used in human in the loops such that until human approves or rejects or give input we will pass the execution

so Human in the state is implemented using Interupts and checkpointers in langraph 

### Command: Combining State Update + Routing
for conditional routing we will have a routing node to decide which node to go
but instead in the previous node only we can decide which node to go 

``` 
return Command(
    update={
        "status": "approved"
    },
    goto="execute"
)
```